In [1]:
from sentence_transformers import util
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# CREATING DOCUMENTS

In [4]:
documents = [
    "Google OAuth allows users to sign in using their Google account.",
    "PostgreSQL indexes make database queries faster.",
    "Supabase Storage is used to store files and images.",
    "JWT tokens can be used to authenticate API requests.",
    "Docker packages applications into portable containers."
]


In [5]:
embeddings = model.encode(documents)

question = "How can I let users log in with Google?"

question_embedding = model.encode(question)

In [6]:
print(question_embedding.shape)

(384,)


In [7]:
scores = util.cos_sim(question_embedding, embeddings)[0]

print(scores)

best_match = scores.argmax()

tensor([0.7329, 0.1152, 0.0516, 0.2679, 0.0672])


In [8]:

print("Most relevant document:")
print(documents[best_match])
print("Similarity score:", scores[best_match])

Most relevant document:
Google OAuth allows users to sign in using their Google account.
Similarity score: tensor(0.7329)


In [9]:
top_k = 3

top_results = scores.argsort(descending=True)[:top_k]

for i in top_results:
    print("Score:", round(scores[i].item(), 4))
    print("Document:", documents[i])
    print()

Score: 0.7329
Document: Google OAuth allows users to sign in using their Google account.

Score: 0.2679
Document: JWT tokens can be used to authenticate API requests.

Score: 0.1152
Document: PostgreSQL indexes make database queries faster.



ATTEMPTING CHUNK CREATION FROM SOURCE TEXT BODY


In [10]:
chunks = [
    """Supabase provides authentication services for web applications.
Users can sign in using email and password or through OAuth providers
such as Google, GitHub, and Discord.""",

    """Supabase Auth uses JSON Web Tokens (JWTs) to manage authenticated
sessions. These tokens can be used to identify users when making
requests to the application backend.""",

    """Row Level Security (RLS) can be used to control which database rows
authenticated users are allowed to access."""
]

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print()

Chunk 1:
Supabase provides authentication services for web applications.
Users can sign in using email and password or through OAuth providers
such as Google, GitHub, and Discord.

Chunk 2:
Supabase Auth uses JSON Web Tokens (JWTs) to manage authenticated
sessions. These tokens can be used to identify users when making
requests to the application backend.

Chunk 3:
Row Level Security (RLS) can be used to control which database rows
authenticated users are allowed to access.



In [3]:
text = """
Supabase provides authentication services.
Users can sign in using email and password.
Users can also sign in through Google OAuth.
OAuth authentication uses access tokens.
These tokens help identify authenticated users.
Supabase can also use Row Level Security.
RLS controls which database rows users can access.
"""

In [4]:
sentences = [s.strip() for s in text.split(".") if s.strip()]

chunks = []

chunk_size = 3
overlap = 1

for i in range(0, len(sentences), chunk_size - overlap):
    chunk = sentences[i:i + chunk_size]
    chunks.append(". ".join(chunk) + ".")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print()

Chunk 1:
Supabase provides authentication services. Users can sign in using email and password. Users can also sign in through Google OAuth.

Chunk 2:
Users can also sign in through Google OAuth. OAuth authentication uses access tokens. These tokens help identify authenticated users.

Chunk 3:
These tokens help identify authenticated users. Supabase can also use Row Level Security. RLS controls which database rows users can access.

Chunk 4:
RLS controls which database rows users can access.



In [5]:
chunk_embeddings = model.encode(chunks)

print(chunk_embeddings.shape)

(4, 384)


In [6]:
question = "How can users sign in with Google?"

question_embedding = model.encode(question)

In [7]:

scores = util.cos_sim(question_embedding, chunk_embeddings)[0]

print(scores)

tensor([0.5636, 0.7547, 0.1853, 0.1245])


In [8]:
top_k = 3

top_results = scores.argsort(descending=True)[:top_k]

for i in top_results:
    print("Score:", round(scores[i].item(), 4))
    print("Document:", chunks[i])
    print()

Score: 0.7547
Document: Users can also sign in through Google OAuth. OAuth authentication uses access tokens. These tokens help identify authenticated users.

Score: 0.5636
Document: Supabase provides authentication services. Users can sign in using email and password. Users can also sign in through Google OAuth.

Score: 0.1853
Document: These tokens help identify authenticated users. Supabase can also use Row Level Security. RLS controls which database rows users can access.



In [9]:
top_k = 2

top_results = scores.argsort(descending=True)[:top_k]

retrieved_chunks = [chunks[i] for i in top_results]

for i, chunk in enumerate(retrieved_chunks):
    print(f"Retrieved Chunk {i+1}:")
    print(chunk)
    print()

Retrieved Chunk 1:
Users can also sign in through Google OAuth. OAuth authentication uses access tokens. These tokens help identify authenticated users.

Retrieved Chunk 2:
Supabase provides authentication services. Users can sign in using email and password. Users can also sign in through Google OAuth.



In [10]:
context = "\n\n".join(retrieved_chunks)

print(context)

Users can also sign in through Google OAuth. OAuth authentication uses access tokens. These tokens help identify authenticated users.

Supabase provides authentication services. Users can sign in using email and password. Users can also sign in through Google OAuth.


In [11]:
prompt = """You are a helpful assistant.
Answer the user's question using the provided context."""

In [12]:
print("QUESTION:")
print(question)

print("\nCONTEXT:")
print(context)


QUESTION:
How can users sign in with Google?

CONTEXT:
Users can also sign in through Google OAuth. OAuth authentication uses access tokens. These tokens help identify authenticated users.

Supabase provides authentication services. Users can sign in using email and password. Users can also sign in through Google OAuth.


In [23]:
prompt = f"""
You are a helpful technical assistant.

Answer the user's question using the provided context.
Give a clear and concise explanation.
Include the relevant details from the context rather than giving only
a one-sentence answer.
Do not add information that is not supported by the context.

Context:
{context}

Question:
{question}

Answer:
"""

In [22]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [14]:
from google import genai

NOW WE WILL CALL THE GEMINI API KEY TO CONNECT OUR PROMPT TO AN LLM , TO COMPLETE THE RAG PIPELINE

In [25]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

CREATING THE GEMINI CLIENT

In [ ]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

SENDING OUR RAG PROMPT

In [19]:
import os

print("Key exists:", os.getenv("GEMINI_API_KEY") is not None)
print("Key length:", len(os.getenv("GEMINI_API_KEY", "")))

Key exists: True
Key length: 39


In [25]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=prompt
)

print(response.text)

Users can sign in through Google OAuth, which is supported by Supabase's authentication services. During this process, the OAuth authentication uses access tokens to help identify authenticated users.
